In [1]:
import os

In [2]:
os.getcwd()

'c:\\Users\\harsh\\Desktop\\text-summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
os.getcwd()

'c:\\Users\\harsh\\Desktop\\text-summarizer'

In [5]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class datatransformationConfig:
    root_dir: Path 
    data_path: Path
    tokenizer_name: Path


In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml,creat_directories

In [10]:
class ConfigurationManager:
    def __init__(self,
                config_filepath=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH):

        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        creat_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> datatransformationConfig:
        config=self.config.data_transformation
        creat_directories([config.root_dir])

        data_transformation_config= datatransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name= config.tokenizer_name,
            )
        return data_transformation_config

In [7]:
import os
from textsummarizer.logging import logger
from datasets import load_dataset,load_from_disk
from transformers import AutoTokenizer


c:\Users\harsh\Desktop\text-summarizer\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

class DataTransformation:
    def __init__(self,config: datatransformationConfig):
        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self,example_batch):
        input_encodings=self.tokenizer(example_batch["dialogue"],max_length=1024,truncation=True)
        target_encodings=self.tokenizer(example_batch["summary"],max_length=128,truncation=True)

        return {
      "input_ids": input_encodings['input_ids'],
      "attention_mask": input_encodings["attention_mask"],
      "labels": target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum=load_from_disk(self.config.data_path)
        dataset_samsum_pt=dataset_samsum.map(self.convert_examples_to_features,batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

    

In [11]:
try:
    config= ConfigurationManager()
    data_transformation_config=config.get_data_transformation_config()
    data_transformation=DataTransformation(config=data_transformation_config)
    data_transformation.convert()


except Exception as e:
    raise e

[2026-10-10 12:50:03,910: INFO: common: yaml file: config\config.yaml loaded succesfully:]
[2026-10-10 12:50:03,916: INFO: common: yaml file: params.yaml loaded succesfully:]
[2026-10-10 12:50:03,917: INFO: common: create director at : artifacts:]
[2026-10-10 12:50:03,918: INFO: common: create director at : artifacts/data_transformation:]
[2026-10-10 12:50:04,404: INFO: _client: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK":]
[2026-10-10 12:50:04,668: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK":]
[2026-10-10 12:50:04,924: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 307 Temporary Redirect":]


[2026-10-10 12:50:04,925: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.:]
[2026-10-10 12:50:04,948: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK":]
[2026-10-10 12:50:04,975: INFO: _client: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK":]


c:\Users\harsh\Desktop\text-summarizer\env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harsh\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[2026-10-10 12:50:05,265: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect":]
[2026-10-10 12:50:05,291: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Ftokenizer_config.json=&etag=%220a6130bfb0bfa44a3a376fe225b327e7ba8fa889%22 "HTTP/1.1 200 OK":]
[2026-10-10 12:50:05,317: INFO: _client: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Ftokenizer_config.json=&etag=%220a6130bfb0bfa44a3a376fe225b327e7ba8fa889%22 "HTTP/1.1 200 OK":]
[2026-10-10 12

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 187790.95 examples/s]
